# IDH Mutation Prediction in Glioma — Full Pipeline
### Beyond Histological Grading: Radiomics, CNN, and Hybrid Fusion

**Datasets:** UCSF-PDGM (training) · EGD (external validation) · TCGA BraTS 2021 (external validation)

**Tracks:** Track A — 7 sequences (T1, T1c, T2, FLAIR, ADC, FA, MD) · Track B — 4 sequences (T1, T1c, T2, FLAIR)

**Arms:** Radiomics (PyRadiomics + SVM/XGBoost) · CNN (ResNet18) · Hybrid Fusion

---
**Cell index:**
- Cell 1: Setup — install dependencies, verify datasets mounted
- Cell 2: Labels — build patient list, binarize IDH
- Cell 3: Splits — stratified 5-fold, unit test
- Cell 4: Preprocessing — clip, normalize, build CNN tensors + summary
- Cell 5: CNN training — Track A (ResNet18 7-channel)
- Cell 6: CNN training — Track B (ResNet18 4-channel)
- Cell 7: Radiomics feature extraction
- Cell 8: Radiomics feature selection + classification (SVM/XGBoost) + SHAP
- Cell 9: Hybrid fusion — embedding extraction + PCA + concatenation
- Cell 10: Fusion classifier (SVM/XGBoost) — early and late fusion
- Cell 11: External validation — EGD + TCGA (Track B models only)
- Cell 12: Ablation studies — 4-seq vs 5-seq vs 7-seq
- Cell 13: Statistical tests — DeLong, bootstrap CI, Bonferroni
- Cell 14: Figures — ROC curves, SHAP, Grad-CAM, confusion matrices
- Cell 15: Results table + final summary

## Cell 1 — Setup
**Run every new session before anything else.**

In [1]:
!pip install nibabel SimpleITK scikit-image -q

import os, json, glob
import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
from skimage.transform import resize
from sklearn.model_selection import StratifiedKFold

WORKING = '/kaggle/working'
os.makedirs(f'{WORKING}/trackA_slices', exist_ok=True)
# Track B derived at runtime from trackA[:, :4, :, :] — no separate directory needed

# Input paths — all 10 batch datasets
KAGGLE_USER  = 'adesaladaniel'
BATCH_ROOTS  = [f'/kaggle/input/datasets/{KAGGLE_USER}/ucsf-pdgm-batch-{i:02d}' for i in range(1, 11)]

# Sequence filenames
TRACK_A_SEQS = ['T1', 'T1c', 'T2', 'FLAIR', 'ADC', 'DTI_eddy_FA', 'DTI_eddy_MD']
TRACK_B_SEQS = ['T1', 'T1c', 'T2', 'FLAIR']
MASK_SEQS    = ['tumor_segmentation', 'brain_segmentation']

print('Dependencies loaded.')
print(f'Batch roots found: {sum(os.path.exists(b) for b in BATCH_ROOTS)}/10')
for b in BATCH_ROOTS:
    status = 'OK     ' if os.path.exists(b) else 'MISSING'
    print(f'  {status}  {b}')

Dependencies loaded.
Batch roots found: 10/10
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-02
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-03
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-04
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-05
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-06
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-07
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-08
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-09
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10


## Cell 2 — Labels
**build patient list, binarize IDH**

In [2]:
import requests

# ── DISCOVER ALL PATIENT FOLDERS ─────────────────────────────────────────────
patient_dirs = {}
for batch_root in BATCH_ROOTS:
    if not os.path.exists(batch_root):
        continue
    for entry in sorted(os.listdir(batch_root)):
        if entry.startswith('UCSF-PDGM-'):
            full_path = os.path.join(batch_root, entry)
            if os.path.isdir(full_path):
                patient_dirs[entry] = full_path

print(f'Patient folders found: {len(patient_dirs)}')
if len(patient_dirs) == 0:
    print('ERROR: No patient folders found.')
    raise SystemExit

# Print first 3 and last 3 to verify structure
sample = sorted(patient_dirs.keys())
for pid in sample[:3] + sample[-3:]:
    print(f'  {pid}  →  {patient_dirs[pid]}')

print()

# ── LOAD METADATA AND BINARIZE IDH ───────────────────────────────────────────
meta_path = f'{WORKING}/meta.csv'
if not os.path.exists(meta_path):
    print('Downloading metadata CSV...')
    r = requests.get('https://www.cancerimagingarchive.net/wp-content/uploads/UCSF-PDGM-metadata.csv')
    with open(meta_path, 'wb') as f:
        f.write(r.content)
    print('Downloaded.')

df_meta = pd.read_csv(meta_path)

# Standardize patient ID to UCSF-PDGM-XXXX
df_meta['ID_padded'] = df_meta['ID'].apply(
    lambda p: f'UCSF-PDGM-{int(str(p).split("-")[-1]):04d}'
)

# Binarize IDH: wildtype=0, all mutation variants=1
def binarize_idh(val):
    if pd.isna(val):
        return None
    v = str(val).strip().lower()
    if v == 'wildtype':
        return 0
    return 1

df_meta['IDH_binary'] = df_meta['IDH'].apply(binarize_idh)

# Exclude follow-up scans
FOLLOWUP_IDS = [
    'UCSF-PDGM-0433', 'UCSF-PDGM-0431', 'UCSF-PDGM-0396',
    'UCSF-PDGM-0429', 'UCSF-PDGM-0409', 'UCSF-PDGM-0391'
]
df_labels = df_meta[
    df_meta['IDH_binary'].notna() &
    ~df_meta['ID_padded'].isin(FOLLOWUP_IDS)
][['ID_padded', 'IDH', 'IDH_binary', 'WHO CNS Grade',
   'Final pathologic diagnosis (WHO 2021)', 'MGMT status']].copy()
df_labels = df_labels.reset_index(drop=True)
df_labels.rename(columns={'ID_padded': 'patient_id'}, inplace=True)

# Keep only patients we have folders for
df_labels = df_labels[df_labels['patient_id'].isin(patient_dirs)].reset_index(drop=True)
df_labels['folder_path'] = df_labels['patient_id'].map(patient_dirs)

wt = (df_labels['IDH_binary'] == 0).sum()
mt = (df_labels['IDH_binary'] == 1).sum()
print(f'Patients with labels + folders : {len(df_labels)}  (expect 495)')
print(f'IDH Wildtype                   : {wt}')
print(f'IDH Mutant                     : {mt}')
print(f'Class ratio                    : {wt/mt:.2f}:1')
print()
print('Grade distribution:')
print(df_labels['WHO CNS Grade'].value_counts().sort_index().to_string())

df_labels.to_csv(f'{WORKING}/labels.csv', index=False)
print(f'\nSaved: {WORKING}/labels.csv')

Patient folders found: 495
  UCSF-PDGM-0004  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01/UCSF-PDGM-0004
  UCSF-PDGM-0005  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01/UCSF-PDGM-0005
  UCSF-PDGM-0007  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01/UCSF-PDGM-0007
  UCSF-PDGM-0539  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10/UCSF-PDGM-0539
  UCSF-PDGM-0540  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10/UCSF-PDGM-0540
  UCSF-PDGM-0541  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10/UCSF-PDGM-0541

Downloaded.
Patients with labels + folders : 495  (expect 495)
IDH Wildtype                   : 392
IDH Mutant                     : 103
Class ratio                    : 3.81:1

Grade distribution:
WHO CNS Grade
2     56
3     43
4    396

Saved: /kaggle/working/labels.csv


## Cell 3: Splits 
**stratified 5-fold, unit test**

In [3]:
# ── STRATIFIED 5-FOLD SPLIT ───────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

patient_ids = df_labels['patient_id'].values
idh_labels  = df_labels['IDH_binary'].values.astype(int)
fold_assignments = np.zeros(len(df_labels), dtype=int)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(patient_ids, idh_labels)):
    fold_assignments[val_idx] = fold_idx

df_labels['fold'] = fold_assignments
df_labels.to_csv(f'{WORKING}/labels.csv', index=False)

# Save fold assignments as JSON
splits = {}
for fold in range(5):
    fold_patients = df_labels[df_labels['fold'] == fold]['patient_id'].tolist()
    splits[f'fold_{fold}'] = fold_patients

with open(f'{WORKING}/patient_splits.json', 'w') as f:
    json.dump(splits, f, indent=2)

# ── UNIT TEST: ZERO PATIENT OVERLAP ──────────────────────────────────────────
print('Running data leakage unit test...')
print()
all_passed = True
for fold in range(5):
    val_patients   = set(splits[f'fold_{fold}'])
    train_patients = set()
    for other_fold in range(5):
        if other_fold != fold:
            train_patients.update(splits[f'fold_{other_fold}'])
    overlap = val_patients & train_patients
    if overlap:
        print(f'  FAIL fold {fold}: {len(overlap)} patients in both train and val!')
        all_passed = False
    else:
        val_wt = sum(1 for p in val_patients
                     if df_labels.loc[df_labels['patient_id']==p, 'IDH_binary'].values[0] == 0)
        val_mt = len(val_patients) - val_wt
        print(f'  PASS fold {fold}: {len(val_patients)} patients — {val_wt} wildtype / {val_mt} mutant')

print()
if all_passed:
    print('All folds pass — zero patient ID overlap confirmed.')
    print(f'Saved: {WORKING}/patient_splits.json')
else:
    print('FAILED — do not proceed.')
    raise SystemExit

Running data leakage unit test...

  PASS fold 0: 99 patients — 79 wildtype / 20 mutant
  PASS fold 1: 99 patients — 79 wildtype / 20 mutant
  PASS fold 2: 99 patients — 78 wildtype / 21 mutant
  PASS fold 3: 99 patients — 78 wildtype / 21 mutant
  PASS fold 4: 99 patients — 78 wildtype / 21 mutant

All folds pass — zero patient ID overlap confirmed.
Saved: /kaggle/working/patient_splits.json


## Cell 4: Preprocessing 
**clip, normalize, build CNN tensors**

In [ ]:
# ── CONSTANTS ─────────────────────────────────────────────────────────────────
TRACK_A_SEQS  = ['T1', 'T1c', 'T2', 'FLAIR', 'ADC', 'DTI_eddy_FA', 'DTI_eddy_MD']
TUMOR_SEQ     = 'tumor_segmentation'
BRAIN_SEQ     = 'brain_segmentation'
SLICE_SIZE    = 224
TUMOR_THR     = 0.001

# ── RELOAD STATE ──────────────────────────────────────────────────────────────
df_labels     = pd.read_csv(f'{WORKING}/labels.csv')
progress_path = f'{WORKING}/progress.json'

if os.path.exists(progress_path):
    with open(progress_path) as f:
        progress = json.load(f)
    completed_raw = set(progress.get('completed', []))
    completed = set(
        p for p in completed_raw
        if os.path.exists(f'{WORKING}/trackA_slices/{p}.npy')
    )
    lost = completed_raw - completed
    if lost:
        print(f'WARNING: {len(lost)} patients marked done but .npy files missing — will reprocess')
    failed     = progress.get('failed', {})
    # Only keep slice_meta for patients confirmed on disk
    slice_meta = [s for s in progress.get('slice_meta', [])
                  if s['patient_id'] in completed]
    print(f'RESUMING — {len(completed)} confirmed on disk, {len(failed)} failed')
else:
    completed  = set()
    failed     = {}
    slice_meta = []
    print('Starting fresh')

# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────
def get_nii_path(patient_dir, patient_id, seq_name):
    """
    Handles nested folder structure:
    UCSF-PDGM-0004/
      UCSF-PDGM-0004_T1.nii/
        UCSF-PDGM-0004_T1.nii  <-- actual file
    Also handles flat structure as fallback.
    """
    filename = f'{patient_id}_{seq_name}.nii'
    nested   = os.path.join(patient_dir, filename, filename)
    flat     = os.path.join(patient_dir, filename)
    if os.path.isfile(nested):
        return nested
    elif os.path.isfile(flat):
        return flat
    return None

def load_volume(path):
    img = nib.load(path)
    return img.get_fdata(dtype=np.float32)

def clip_and_normalize(vol, brain_mask):
    brain_voxels = vol[brain_mask > 0]
    if brain_voxels.size == 0:
        return vol
    # Clip to 1st-99th percentile within brain mask
    p1  = np.percentile(brain_voxels, 1)
    p99 = np.percentile(brain_voxels, 99)
    vol = np.clip(vol, p1, p99)
    # Z-score normalize within brain mask
    brain_voxels = vol[brain_mask > 0]
    mean = brain_voxels.mean()
    std  = brain_voxels.std()
    if std < 1e-8:
        vol = np.zeros_like(vol)
    else:
        vol = (vol - mean) / std
    vol[brain_mask == 0] = 0.0
    return vol

def select_tumor_slices(tumor_mask):
    n_slices   = tumor_mask.shape[2]
    slice_area = tumor_mask.shape[0] * tumor_mask.shape[1]
    selected   = []
    for s in range(n_slices):
        coverage = (tumor_mask[:, :, s] > 0).sum() / slice_area
        if coverage >= TUMOR_THR:
            selected.append(s)
    return selected

def resize_slice(slc):
    return resize(slc, (SLICE_SIZE, SLICE_SIZE),
                  order=1, mode='constant',
                  anti_aliasing=True,
                  preserve_range=True).astype(np.float32)

def preprocess_patient(patient_id, patient_dir):
    try:
        # Resolve file paths
        tumor_path = get_nii_path(patient_dir, patient_id, TUMOR_SEQ)
        brain_path = get_nii_path(patient_dir, patient_id, BRAIN_SEQ)

        if tumor_path is None:
            return None, 0, f'Cannot find tumor_segmentation.nii'
        if brain_path is None:
            return None, 0, f'Cannot find brain_segmentation.nii'

        # Load and binarize masks
        tumor_mask = load_volume(tumor_path)
        brain_mask = load_volume(brain_path)
        tumor_mask = (tumor_mask > 0).astype(np.float32)
        brain_mask = (brain_mask > 0).astype(np.float32)

        # Select tumor-containing axial slices
        selected = select_tumor_slices(tumor_mask)
        if len(selected) == 0:
            return None, 0, 'No tumor slices above 0.1% coverage threshold'

        # Load, normalize, extract slices for each sequence
        channel_arrays = []
        for seq in TRACK_A_SEQS:
            seq_path = get_nii_path(patient_dir, patient_id, seq)
            if seq_path is None:
                return None, 0, f'Cannot find {seq}.nii'
            vol = load_volume(seq_path)
            vol = clip_and_normalize(vol, brain_mask)
            seq_slices = np.stack(
                [resize_slice(vol[:, :, s]) for s in selected],
                axis=0
            )  # shape: (n_slices, 224, 224)
            channel_arrays.append(seq_slices)

        # Stack into Track A tensor: (n_slices, 7, 224, 224)
        # Track B = trackA[:, :4, :, :] — derived at runtime, not saved separately
        trackA = np.stack(channel_arrays, axis=1).astype(np.float16)

        return trackA, len(selected), None

    except Exception as e:
        return None, 0, str(e)

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
remaining = [p for p in df_labels['patient_id'].tolist() if p not in completed]

print(f'Patients remaining : {len(remaining)}')
print(f'Saving to          : {WORKING}/trackA_slices/')
print(f'Track B derived at runtime from first 4 channels of Track A')
print('='*60)

for i, patient_id in enumerate(remaining):
    row         = df_labels[df_labels['patient_id'] == patient_id].iloc[0]
    patient_dir = row['folder_path']
    fold        = int(row['fold'])
    idh_label   = int(row['IDH_binary'])

    trackA, n_slices, error = preprocess_patient(patient_id, patient_dir)

    if error:
        failed[patient_id] = error
        print(f'  [{i+1}/{len(remaining)}] FAIL {patient_id} — {error}')
    else:
        np.save(f'{WORKING}/trackA_slices/{patient_id}.npy', trackA)
        for s in range(n_slices):
            slice_meta.append({
                'patient_id': patient_id,
                'slice_idx':  s,
                'fold':       fold,
                'idh_label':  idh_label
            })
        completed.add(patient_id)
        # Remove from failed if it was previously failed
        failed.pop(patient_id, None)
        print(f'  [{i+1}/{len(remaining)}] OK  {patient_id} — {n_slices} slices | fold {fold} | IDH {idh_label}')

    # Save progress after every patient
    with open(progress_path, 'w') as f:
        json.dump({
            'completed':  list(completed),
            'failed':     failed,
            'slice_meta': slice_meta
        }, f)

print()
print('='*60)
print(f'Completed : {len(completed)}/{len(df_labels)}')
print(f'Failed    : {len(failed)}')
if failed:
    print('Failed patients:')
    for pid, reason in failed.items():
        print(f'  {pid}: {reason}')
if len(completed) == len(df_labels):
    print('All patients done — proceed to Cell 5.')
else:
    print('Re-run Cell 1 then Cell 4 to continue.')



# ── SUMMARY ───────────────────────────────────────────────────────────────────
df_slices = pd.DataFrame(slice_meta)
df_slices.to_csv(f'{WORKING}/slice_metadata.csv', index=False)

print()
print('='*60)
print('PREPROCESSING SUMMARY')
print('='*60)
print(f'Patients processed     : {len(completed)}')
print(f'Patients failed        : {len(failed)}')
print(f'Total slices           : {len(df_slices)}')
print(f'Average slices/patient : {len(df_slices)/len(completed):.1f}')
print()
print('Slices per fold:')
for fold in range(5):
    fold_df = df_slices[df_slices['fold'] == fold]
    wt = (fold_df['idh_label'] == 0).sum()
    mt = (fold_df['idh_label'] == 1).sum()
    print(f'  Fold {fold}: {len(fold_df):>5} slices — {wt} wildtype / {mt} mutant')
print()
print('Files in /kaggle/working/:')
print('  labels.csv          — patient ID, IDH label, grade, fold')
print('  patient_splits.json — fold assignments per patient')
print('  slice_metadata.csv  — slice to patient mapping')
print('  trackA_slices/      — 495 .npy files, float16, (n,7,224,224)')
print('  Track B = trackA[:, :4, :, :] derived at runtime')

Starting fresh
Patients remaining : 495
Saving to          : /kaggle/working/trackA_slices/
Track B derived at runtime from first 4 channels of Track A
  [1/495] OK  UCSF-PDGM-0004 — 42 slices | fold 0 | IDH 0
  [2/495] OK  UCSF-PDGM-0005 — 51 slices | fold 3 | IDH 0
  [3/495] OK  UCSF-PDGM-0007 — 77 slices | fold 3 | IDH 0
  [4/495] OK  UCSF-PDGM-0008 — 75 slices | fold 0 | IDH 0
  [5/495] OK  UCSF-PDGM-0009 — 63 slices | fold 4 | IDH 0
  [6/495] OK  UCSF-PDGM-0010 — 77 slices | fold 2 | IDH 0
  [7/495] OK  UCSF-PDGM-0011 — 70 slices | fold 0 | IDH 0
  [8/495] OK  UCSF-PDGM-0012 — 71 slices | fold 2 | IDH 0
  [9/495] OK  UCSF-PDGM-0013 — 57 slices | fold 1 | IDH 0
  [10/495] OK  UCSF-PDGM-0014 — 85 slices | fold 1 | IDH 0
  [11/495] OK  UCSF-PDGM-0015 — 63 slices | fold 0 | IDH 0
  [12/495] OK  UCSF-PDGM-0016 — 60 slices | fold 2 | IDH 0
  [13/495] OK  UCSF-PDGM-0017 — 59 slices | fold 3 | IDH 0
  [14/495] OK  UCSF-PDGM-0018 — 32 slices | fold 2 | IDH 0
  [15/495] OK  UCSF-PDGM-0019 —

In [6]:
# ── CELL 4 AUDIT: Check slice threshold totals ───────────────────────────────

import os, glob
import numpy as np
import pandas as pd
import nibabel as nib

WORKING = '/kaggle/working'

df_labels = pd.read_csv(f'{WORKING}/labels.csv')

def get_nii_path(patient_dir, patient_id, seq_name):
    filename = f'{patient_id}_{seq_name}.nii'
    nested   = os.path.join(patient_dir, filename, filename)
    flat     = os.path.join(patient_dir, filename)
    if os.path.isfile(nested):
        return nested
    elif os.path.isfile(flat):
        return flat
    return None

thresholds = {
    '5%':    0.05,
    '1%':    0.01,
    '0.5%':  0.005,
    '0.1%':  0.001,
    '0.05%': 0.0005,
    '0.01%': 0.0001,
}

totals = {k: 0 for k in thresholds}
zero_patients = {k: 0 for k in thresholds}

for _, row in df_labels.iterrows():
    pid = row['patient_id']
    patient_dir = row['folder_path']
    tumor_path = get_nii_path(patient_dir, pid, 'tumor_segmentation')
    
    if tumor_path is None:
        print(f'Missing tumor mask: {pid}')
        continue
    
    mask = nib.load(tumor_path).get_fdata(dtype=np.float32)
    mask = (mask > 0)
    
    slice_area = mask.shape[0] * mask.shape[1]
    coverages = []
    
    for s in range(mask.shape[2]):
        cov = mask[:, :, s].sum() / slice_area
        coverages.append(cov)
    
    coverages = np.array(coverages)
    
    for name, thr in thresholds.items():
        count = int((coverages >= thr).sum())
        totals[name] += count
        if count == 0:
            zero_patients[name] += 1

print('='*60)
print('SLICE SELECTION AUDIT')
print('='*60)
print(f'Patients checked: {len(df_labels)}')
print()
for name in thresholds:
    print(
        f'Threshold {name:>5}: '
        f'{totals[name]:>6} total slices | '
        f'{totals[name]/len(df_labels):>5.1f} avg/patient | '
        f'{zero_patients[name]} patients with zero slices'
    )

print()
print('='*60)
print('Saved .npy tensor check')
print('='*60)

slices_dir = f'{WORKING}/trackA_slices'
npy_files = sorted(glob.glob(f'{slices_dir}/*.npy'))
saved_total = 0
saved_shapes = []

for f in npy_files:
    arr = np.load(f, mmap_mode='r')
    saved_total += arr.shape[0]
    saved_shapes.append(arr.shape)

print(f'.npy files found: {len(npy_files)}')
print(f'Saved total slices: {saved_total}')
print(f'Example shape: {saved_shapes[0] if saved_shapes else None}')

SLICE SELECTION AUDIT
Patients checked: 495

Threshold    5%:   3889 total slices |   7.9 avg/patient | 332 patients with zero slices
Threshold    1%:  21901 total slices |  44.2 avg/patient | 16 patients with zero slices
Threshold  0.5%:  25324 total slices |  51.2 avg/patient | 6 patients with zero slices
Threshold  0.1%:  28881 total slices |  58.3 avg/patient | 0 patients with zero slices
Threshold 0.05%:  29621 total slices |  59.8 avg/patient | 0 patients with zero slices
Threshold 0.01%:  30340 total slices |  61.3 avg/patient | 0 patients with zero slices

Saved .npy tensor check
.npy files found: 495
Saved total slices: 28881
Example shape: (42, 7, 224, 224)


## Cell 5: CNN Training

**Track A (ResNet18 7-channel)**

In [5]:
# ── CELL 5: CNN TRAINING — TRACK A (ResNet18 7-channel) ──────────────────────
# Permanent save: model weights + progress committed to glioma-idh-models
# after every fold. Resume-safe — skips completed folds on restart.
# To resume: run Cell 1 → Cell 4 → Cell 5. Folds already done are skipped.
#
# Metrics: AUC, accuracy, sensitivity, specificity, F1, Brier score
# Patient-level AUC (soft voting) is the primary metric
# Per-patient normalization confirmed (no cross-fold leakage)
#
# FIXES APPLIED:
# 1. Lazy mmap loading — prevents RAM OOM on Kaggle P100
# 2. Best-epoch re-evaluation — patient predictions from best weights, not last epoch
# 3. Cumulative Kaggle commit — all completed fold weights saved together
# 4. best_metrics initialized defensively

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
import torchvision.transforms.functional as TF
import numpy as np
import pandas as pd
import json, os, shutil, subprocess
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss
)
from torch.optim.lr_scheduler import CosineAnnealingLR
from kaggle_secrets import UserSecretsClient
from kaggle.api.kaggle_api_extended import KaggleApi

WORKING = '/kaggle/working'
os.makedirs(f'{WORKING}/cnn_trackA', exist_ok=True)
os.makedirs(f'{WORKING}/trackA_slices', exist_ok=True)

# ── KAGGLE API SETUP ──────────────────────────────────────────────────────────
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')
kaggle_api = KaggleApi()
kaggle_api.authenticate()
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'
print(f'Kaggle API ready — saving to {MODELS_DATASET}')

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 7          # Track A — 7 sequences
N_CLASSES    = 2
BATCH_SIZE   = 32
MAX_EPOCHS   = 50
PATIENCE     = 10         # early stopping on patient-level AUC
LR_HEAD      = 1e-3
LR_BACKBONE  = 1e-4
WEIGHT_DECAY = 1e-4
CLASS_WEIGHT = 3.81       # confirmed from actual UCSF-PDGM data
SEED         = 42
N_FOLDS      = 5
SAVE_DIR     = f'{WORKING}/cnn_trackA'
os.makedirs(SAVE_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f'Device    : {DEVICE}')
print(f'Track     : A — {N_CHANNELS} channels')
print(f'Class wt  : {CLASS_WEIGHT}')

# ── PERMANENT SAVE FUNCTION ───────────────────────────────────────────────────
def commit_to_kaggle(local_dir, message):
    """Commits everything in local_dir to glioma-idh-models."""
    metadata_path = os.path.join(local_dir, 'dataset-metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump({
            'title':     'Glioma IDH Models',
            'id':        MODELS_DATASET,
            'licenses':  [{'name': 'other'}],
            'isPrivate': True
        }, f)
    result = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', local_dir, '-m', message, '--dir-mode', 'zip'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'  Saved to Kaggle: {message} ✓')
    else:
        print(f'  WARNING: Kaggle save failed: {result.stderr[:200]}')
    return result.returncode == 0

# ── PERMANENT RESUME FUNCTION ─────────────────────────────────────────────────
def download_from_kaggle(filename, local_path):
    """Downloads a file from glioma-idh-models."""
    try:
        result = subprocess.run(
            ['kaggle', 'datasets', 'download',
             MODELS_DATASET, '--file', filename,
             '--path', os.path.dirname(local_path), '--unzip'],
            capture_output=True, text=True
        )
        return result.returncode == 0 and os.path.exists(local_path)
    except Exception:
        return False

# ── DATASET (LAZY MMAP LOADING) ──────────────────────────────────────────────
class GliomaDataset(Dataset):
    """
    Memory-mapped dataset — loads only the requested slice from disk.
    Prevents RAM OOM on Kaggle (P100 has ~13GB system RAM).
    
    Each .npy file is memory-mapped; only the specific slice is read
    into RAM when __getitem__ is called. With num_workers=2,
    prefetching keeps the GPU fed without loading everything at once.
    
    Normalization was done per-patient in Cell 4 using each patient's
    own brain mask statistics. No cross-patient or cross-fold leakage.
    """
    def __init__(self, patient_ids, df_labels, slices_dir,
                 n_channels=7, augment=False):
        self.slices_dir  = slices_dir
        self.n_channels  = n_channels
        self.augment     = augment
        self.items       = []   # list of (pid, slice_idx, label)

        label_map = dict(zip(df_labels['patient_id'],
                             df_labels['IDH_binary'].astype(int)))
        for pid in patient_ids:
            npy_path = os.path.join(slices_dir, f'{pid}.npy')
            if not os.path.exists(npy_path):
                continue
            # Memory-map just to read shape — no RAM used
            arr = np.load(npy_path, mmap_mode='r')
            n_slices = arr.shape[0]
            label = label_map[pid]
            for s in range(n_slices):
                self.items.append((pid, s, label))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        pid, s, label = self.items[idx]
        # Memory-map the file, read only slice s
        arr = np.load(
            os.path.join(self.slices_dir, f'{pid}.npy'),
            mmap_mode='r'
        )
        # Convert float16 → float32 for GPU training
        x = torch.tensor(
            arr[s, :self.n_channels].astype(np.float32),
            dtype=torch.float32
        )
        if self.augment:
            # ALL spatial transforms applied to all channels simultaneously
            # (preserves co-registration between sequences)
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])    # horizontal flip
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])    # vertical flip
            if torch.rand(1) > 0.5:
                angle = (torch.rand(1).item() - 0.5) * 30  # ±15 degrees
                x = TF.rotate(x, angle)
            # Intensity scaling — same factor for ALL channels
            scale = 0.9 + torch.rand(1).item() * 0.2
            x = x * scale
            # Gaussian noise
            x = x + torch.randn_like(x) * 0.02
        return x, label, pid

# ── MODEL ─────────────────────────────────────────────────────────────────────
def build_model(n_channels, n_classes):
    """
    ResNet18 adapted for N-channel input.
    Averages the 3 pretrained RGB channel weights → replicates to N channels.
    Classifier head: Linear(512→256) → ReLU → Dropout(0.5) → Linear(256→2)
    """
    model    = resnet18(weights='IMAGENET1K_V1')
    old_conv = model.conv1
    new_conv = nn.Conv2d(n_channels, 64, kernel_size=7,
                         stride=2, padding=3, bias=False)
    with torch.no_grad():
        avg_weight      = old_conv.weight.mean(dim=1, keepdim=True)
        new_conv.weight = nn.Parameter(avg_weight.repeat(1, n_channels, 1, 1))
    model.conv1 = new_conv
    model.fc    = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, n_classes)
    )
    return model

# ── TRAINING ──────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

# ── EVALUATION WITH FULL METRICS ──────────────────────────────────────────────
def evaluate(model, loader, criterion, device, label_map):
    """
    Evaluates at PATIENT level using soft voting (mean slice probabilities).
    Returns patient-level AUC, sensitivity, specificity, F1, accuracy, Brier.
    Patient-level is the clinically meaningful metric — NOT slice-level.
    """
    model.eval()
    total_loss    = 0
    patient_probs = {}
    all_slice_probs  = []
    all_slice_labels = []

    with torch.no_grad():
        for x, y, pids in loader:
            x, y   = x.to(device), y.to(device)
            out    = model(x)
            loss   = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            probs  = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            labels = y.cpu().numpy()
            all_slice_probs.extend(probs)
            all_slice_labels.extend(labels)
            for prob, pid in zip(probs, pids):
                if pid not in patient_probs:
                    patient_probs[pid] = []
                patient_probs[pid].append(float(prob))

    # Slice-level AUC (reference only — not reported in paper)
    slice_auc = roc_auc_score(all_slice_labels, all_slice_probs)

    # Patient-level metrics (soft voting)
    pid_list       = list(patient_probs.keys())
    pid_mean_probs = [np.mean(patient_probs[p]) for p in pid_list]
    pid_labels     = [label_map[p] for p in pid_list]

    patient_auc = roc_auc_score(pid_labels, pid_mean_probs)

    # Binary predictions at 0.5 threshold
    pid_preds = [1 if p >= 0.5 else 0 for p in pid_mean_probs]

    accuracy    = accuracy_score(pid_labels, pid_preds)
    sensitivity = recall_score(pid_labels, pid_preds, zero_division=0)
    f1          = f1_score(pid_labels, pid_preds, zero_division=0)
    brier       = brier_score_loss(pid_labels, pid_mean_probs)

    cm = confusion_matrix(pid_labels, pid_preds, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    avg_loss = total_loss / len(loader.dataset)

    metrics = {
        'patient_auc':  round(float(patient_auc), 4),
        'slice_auc':    round(float(slice_auc), 4),
        'accuracy':     round(float(accuracy), 4),
        'sensitivity':  round(float(sensitivity), 4),
        'specificity':  round(float(specificity), 4),
        'f1':           round(float(f1), 4),
        'brier':        round(float(brier), 4),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)
    }
    return avg_loss, metrics, patient_probs

# ── RELOAD PROGRESS ───────────────────────────────────────────────────────────
cnn_progress_path = f'{WORKING}/cnn_trackA_progress.json'

if not os.path.exists(cnn_progress_path):
    print('Checking Kaggle for saved progress...')
    restored = download_from_kaggle('cnn_trackA_progress.json', cnn_progress_path)
    print('Progress restored from Kaggle ✓' if restored
          else 'No saved progress — starting fresh')

if os.path.exists(cnn_progress_path):
    with open(cnn_progress_path) as f:
        cnn_progress = json.load(f)
    completed_folds = cnn_progress.get('completed_folds', [])
    fold_results    = cnn_progress.get('fold_results', {})
    all_val_preds   = cnn_progress.get('all_val_preds', {})
    print(f'RESUMING CNN Track A — completed folds: {completed_folds}')
else:
    completed_folds = []
    fold_results    = {}
    all_val_preds   = {}
    print('Starting CNN Track A fresh')

# Restore saved model weights from Kaggle for completed folds
for fold in completed_folds:
    weight_file = f'cnn_trackA_fold{fold}_best.pth'
    local_path  = f'{SAVE_DIR}/fold{fold}_best.pth'
    if not os.path.exists(local_path):
        print(f'Restoring weights fold {fold} from Kaggle...')
        download_from_kaggle(weight_file, local_path)

# ── LOAD LABELS AND SPLITS ────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
with open(f'{WORKING}/patient_splits.json') as f:
    splits = json.load(f)

label_map     = dict(zip(df_labels['patient_id'],
                         df_labels['IDH_binary'].astype(int)))
class_weights = torch.tensor([1.0, CLASS_WEIGHT],
                              dtype=torch.float32).to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

# ── 5-FOLD TRAINING LOOP ──────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK A — 5-FOLD CROSS VALIDATION')
print(f'{"="*60}')

for fold in range(N_FOLDS):
    if fold in completed_folds:
        print(f'\nFold {fold} — already done, skipping')
        continue

    print(f'\nFOLD {fold}/{N_FOLDS-1}')
    print('-' * 40)

    val_patients   = splits[f'fold_{fold}']
    train_patients = []
    for other_fold in range(N_FOLDS):
        if other_fold != fold:
            train_patients.extend(splits[f'fold_{other_fold}'])

    print(f'Train patients : {len(train_patients)}')
    print(f'Val patients   : {len(val_patients)}')

    train_ds = GliomaDataset(
        train_patients, df_labels,
        f'{WORKING}/trackA_slices',
        n_channels=N_CHANNELS, augment=True
    )
    val_ds = GliomaDataset(
        val_patients, df_labels,
        f'{WORKING}/trackA_slices',
        n_channels=N_CHANNELS, augment=False
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

    print(f'Train slices   : {len(train_ds)}')
    print(f'Val slices     : {len(val_ds)}')

    # Fresh model for this fold
    model = build_model(N_CHANNELS, N_CLASSES).to(DEVICE)

    optimizer = optim.AdamW([
        {'params': model.fc.parameters(),
         'lr': LR_HEAD},
        {'params': [p for n, p in model.named_parameters()
                    if not n.startswith('fc')],
         'lr': LR_BACKBONE}
    ], weight_decay=WEIGHT_DECAY)

    scheduler    = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    best_val_auc = 0.0
    best_epoch   = 0
    patience_ctr = 0
    best_metrics = {}    # FIX 4: defensive initialization
    history      = []

    # Freeze backbone for first 5 epochs
    for name, param in model.named_parameters():
        if not name.startswith('fc'):
            param.requires_grad = False

    for epoch in range(1, MAX_EPOCHS + 1):

        # Unfreeze backbone after epoch 5
        if epoch == 6:
            for param in model.parameters():
                param.requires_grad = True
            # Re-create optimizer with all params now requiring grad
            optimizer = optim.AdamW([
                {'params': model.fc.parameters(),
                 'lr': LR_HEAD},
                {'params': [p for n, p in model.named_parameters()
                            if not n.startswith('fc')],
                 'lr': LR_BACKBONE}
            ], weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(
                optimizer, T_max=MAX_EPOCHS - 5
            )
            print(f'  Epoch {epoch}: backbone unfrozen, optimizer reset')

        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE
        )
        val_loss, metrics, _ = evaluate(
            model, val_loader, criterion, DEVICE, label_map
        )
        scheduler.step()

        history.append({
            'epoch':      epoch,
            'train_loss': round(float(train_loss), 4),
            'val_loss':   round(float(val_loss), 4),
            **metrics
        })

        print(
            f'  Epoch {epoch:02d} | '
            f'loss: {train_loss:.4f}/{val_loss:.4f} | '
            f'pt_AUC: {metrics["patient_auc"]:.4f} | '
            f'sl_AUC: {metrics["slice_auc"]:.4f} | '
            f'sens: {metrics["sensitivity"]:.4f} | '
            f'spec: {metrics["specificity"]:.4f} | '
            f'F1: {metrics["f1"]:.4f} | '
            f'Brier: {metrics["brier"]:.4f}'
        )

        # Early stopping on patient-level AUC
        if metrics['patient_auc'] > best_val_auc:
            best_val_auc = metrics['patient_auc']
            best_epoch   = epoch
            best_metrics = metrics.copy()
            patience_ctr = 0
            torch.save(model.state_dict(),
                       f'{SAVE_DIR}/fold{fold}_best.pth')
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(
                    f'  Early stopping at epoch {epoch} '
                    f'(best epoch {best_epoch}, '
                    f'best patient AUC {best_val_auc:.4f})'
                )
                break

    # ── FIX 2: RELOAD BEST WEIGHTS AND RE-EVALUATE ───────────────────────────
    # Patient predictions must come from the best epoch, not the last epoch
    print(f'  Reloading best weights (epoch {best_epoch}) for final evaluation...')
    model.load_state_dict(
        torch.load(f'{SAVE_DIR}/fold{fold}_best.pth',
                    map_location=DEVICE)
    )
    _, best_metrics, patient_probs = evaluate(
        model, val_loader, criterion, DEVICE, label_map
    )

    # Save patient predictions from best epoch
    for pid, probs in patient_probs.items():
        all_val_preds[pid] = {
            'mean_prob': float(np.mean(probs)),
            'label':     int(label_map[pid]),
            'fold':      fold
        }

    fold_results[str(fold)] = {
        'best_epoch':   best_epoch,
        'best_auc':     round(best_val_auc, 4),
        'best_metrics': best_metrics,
        'history':      history
    }
    completed_folds.append(fold)

    # Print fold summary
    print(f'\n  Fold {fold} best results (epoch {best_epoch}):')
    print(f'    Patient AUC  : {best_metrics["patient_auc"]:.4f}')
    print(f'    Sensitivity  : {best_metrics["sensitivity"]:.4f}')
    print(f'    Specificity  : {best_metrics["specificity"]:.4f}')
    print(f'    F1 score     : {best_metrics["f1"]:.4f}')
    print(f'    Accuracy     : {best_metrics["accuracy"]:.4f}')
    print(f'    Brier score  : {best_metrics["brier"]:.4f}')
    print(f'    TP:{best_metrics["tp"]} TN:{best_metrics["tn"]} '
          f'FP:{best_metrics["fp"]} FN:{best_metrics["fn"]}')

    # Save progress locally
    progress_data = {
        'completed_folds': completed_folds,
        'fold_results':    fold_results,
        'all_val_preds':   all_val_preds
    }
    with open(cnn_progress_path, 'w') as f:
        json.dump(progress_data, f)

    # ── FIX 3: CUMULATIVE KAGGLE COMMIT ───────────────────────────────────────
    # Copy ALL completed fold weights — not just current fold
    print(f'  Saving all completed folds to Kaggle permanently...')
    commit_dir = f'{WORKING}/kaggle_commit_trackA'
    os.makedirs(commit_dir, exist_ok=True)

    for completed_fold in completed_folds:
        src = f'{SAVE_DIR}/fold{completed_fold}_best.pth'
        dst = f'{commit_dir}/cnn_trackA_fold{completed_fold}_best.pth'
        if os.path.exists(src):
            shutil.copy(src, dst)

    shutil.copy(cnn_progress_path,
                f'{commit_dir}/cnn_trackA_progress.json')
    commit_to_kaggle(
        commit_dir,
        f'CNN Track A fold {fold} complete — AUC {best_val_auc:.4f}'
    )
    shutil.rmtree(commit_dir)

    # Free GPU memory before next fold
    del model, optimizer, scheduler
    del train_ds, val_ds, train_loader, val_loader
    torch.cuda.empty_cache()

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK A — CROSS VALIDATION COMPLETE')
print(f'{"="*60}')

aucs          = [fold_results[str(f)]['best_auc'] for f in range(N_FOLDS)]
sensitivities = [fold_results[str(f)]['best_metrics']['sensitivity']
                 for f in range(N_FOLDS)]
specificities = [fold_results[str(f)]['best_metrics']['specificity']
                 for f in range(N_FOLDS)]
f1s           = [fold_results[str(f)]['best_metrics']['f1']
                 for f in range(N_FOLDS)]
briers        = [fold_results[str(f)]['best_metrics']['brier']
                 for f in range(N_FOLDS)]
accuracies    = [fold_results[str(f)]['best_metrics']['accuracy']
                 for f in range(N_FOLDS)]

print(f'\nPer-fold AUC       : {[round(a, 4) for a in aucs]}')
print(f'Mean AUC           : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print(f'Mean Sensitivity   : {np.mean(sensitivities):.4f} ± {np.std(sensitivities):.4f}')
print(f'Mean Specificity   : {np.mean(specificities):.4f} ± {np.std(specificities):.4f}')
print(f'Mean F1            : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'Mean Accuracy      : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}')
print(f'Mean Brier Score   : {np.mean(briers):.4f} ± {np.std(briers):.4f}')
print(f'\nPatients with predictions : {len(all_val_preds)} (expect 495)')
print(f'All weights saved to      : {MODELS_DATASET}')
print(f'\nNext: Cell 6 — CNN Track B (4-channel)')
print(f'\nNote: Wilcoxon signed-rank tests comparing model pairs')
print(f'will be run in Cell 15 when all models are complete.')
print(f'Fold-level AUC scores saved for that comparison.')

Kaggle API ready — saving to adesaladaniel/glioma-idh-models
Device    : cuda
Track     : A — 7 channels
Class wt  : 3.81
Checking Kaggle for saved progress...
No saved progress — starting fresh
Starting CNN Track A fresh


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()



CNN TRACK A — 5-FOLD CROSS VALIDATION

FOLD 0/4
----------------------------------------
Train patients : 396
Val patients   : 99
Train slices   : 23043
Val slices     : 5838
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 216MB/s]


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
